# Phase 2 · Experiment 2C — Context Length Scaling

### Google Colab notebook (independent)

Research question: **how does sink behaviour change as context length grows?** We build nested-prefix prompts at lengths {128, 512, 1024, 2048, 4096} (early context held fixed, token 0 standardised) and measure the `mean_from_k` sink score. Before each length, memory is **estimated** and any configuration that would not fit the current GPU is **skipped gracefully** — on a free T4, 4096 is auto-detected as needing an A100 / high-RAM runtime and the notebook continues. Hypothesis: sink strength increases with length.

**Runtime:** GPU (free **T4** is enough).

**Prerequisites**
- Phase 1 must have been run with `USE_DRIVE=True`, so `attention_sink_data/` is in your Drive project folder.
- `phase2_utils.py` must be uploaded into the same Drive project folder (upload once; it persists).
- Downloads Qwen3-1.7B. On a T4, expect 4096 (and possibly 2048) to be skipped by the memory guard.

This notebook is self-contained: it can be rerun on its own without executing the other experiments.

---

## 0. Setup

In [ ]:
# --- Colab environment setup -------------------------------------------------
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', *['transformers>=4.51', 'datasets>=2.19', 'accelerate', 'scipy', 'pyarrow']], check=True)

print('In Colab:', IN_COLAB)
import torch
print('CUDA:', torch.cuda.is_available(),
      ('| ' + torch.cuda.get_device_name(0)) if torch.cuda.is_available() else '')
if torch.cuda.is_available():
    print('bf16 native:', torch.cuda.is_bf16_supported(), '(T4=False -> fp16 used)')
else:
    print('*** No GPU. Runtime > Change runtime type > GPU (T4). ***')


In [ ]:
# --- Storage + phase2_utils bootstrap ---------------------------------------
# Point at the SAME Drive project folder Phase 1 used, so Phase 1's raw
# attention and this project's phase2_utils.py are both visible.
USE_DRIVE = True   # must match the Phase 1 setting

import sys
from pathlib import Path
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/attention_sink_project')
elif IN_COLAB:
    BASE = Path('/content/attention_sink_project')
else:
    BASE = Path('.')
BASE.mkdir(parents=True, exist_ok=True)

DATA_ROOT    = str(BASE / 'attention_sink_data')          # Phase 1 output
RESULTS_ROOT = str(BASE / 'results' / 'phase2')           # Phase 2 output
Path(RESULTS_ROOT).mkdir(parents=True, exist_ok=True)

# Locate phase2_utils.py (upload it into BASE once; it persists on Drive).
for c in [BASE, Path('/content'), Path('.')]:
    if (Path(c) / 'phase2_utils.py').exists():
        sys.path.insert(0, str(c)); break
try:
    import phase2_utils as U
    print('phase2_utils loaded from', U.__file__)
except ModuleNotFoundError:
    raise SystemExit('Place phase2_utils.py in ' + str(BASE) + ' (or /content) and re-run this cell.')

print('Phase 1 data :', DATA_ROOT)
print('Phase 2 out  :', RESULTS_ROOT)


## 1. Configuration

In [ ]:
from dataclasses import dataclass, field, asdict
import json, numpy as np, pandas as pd

@dataclass
class Config2C:
    model_name: str = 'Qwen/Qwen3-1.7B'
    lengths: tuple = (128, 512, 1024, 2048, 4096)
    k: int = 4
    mem_safety: float = 2.5      # multiplier on the raw attention estimate
    seed: int = 20240517

cfg = Config2C()
U.set_reproducibility(cfg.seed)
EXP = U.experiment_dir(RESULTS_ROOT, 'experiment2C')
logger = U.get_logger('2C', log_file=str(EXP / 'run.log'))
logger.info('Config: %s', asdict(cfg))

## 2. Load model & build length-controlled prompts

In [ ]:
tokenizer, model = U.load_model(cfg.model_name, dtype='auto', logger=logger)
prefix_id = U.choose_prefix_token_id(tokenizer)
base_text = ' '.join(p['eng'] for p in U.EMBEDDED_PAIRS)   # repeated internally to reach max length
prompts = U.make_length_controlled_prompts(tokenizer, base_text, cfg.lengths, prefix_id)
print('built prompt lengths:', {s: prompts[s].shape[1] for s in cfg.lengths})

## 3. Memory-gated extraction (skip configs that would not fit)

In [ ]:
sink_by_len = {}          # length -> [L,H]
col_by_len  = {}          # length -> [L,H,S] per-query sink column (kept for §5 validation)
skipped = {}
for S in cfg.lengths:
    ok, req, avail = U.can_fit_length(S, model, safety=cfg.mem_safety)
    req_gb = req / 1e9; av_gb = (avail or 0) / 1e9
    if not ok:
        msg = 'needs ~%.1f GB, only ~%.1f GB free -> SKIP (use an A100 / high-RAM runtime)' % (req_gb, av_gb)
        logger.warning('length %d: %s', S, msg)
        skipped[S] = msg
        continue
    logger.info('length %d: est ~%.1f GB (<= ~%.1f GB free) -> extracting', S, req_gb, av_gb)
    col = U.extract_sink_column(prompts[S], model)          # [L,H,S] key-0 column only
    sink_by_len[S] = U.sink_matrix_from_column(col, k=cfg.k) # [L,H]
    col_by_len[S] = col   # retained for the Metric Validation subsection (~3.7 MB at S=2048)

with open(EXP / 'skipped.json', 'w') as f:
    json.dump({str(k): v for k, v in skipped.items()}, f, indent=2)
print('extracted:', sorted(sink_by_len), '| skipped:', sorted(skipped))
assert sink_by_len, 'No length fit in memory; try a high-RAM/A100 runtime or smaller lengths.'

## 4. Curves, summary stats & save

In [ ]:
globals_by_len = {S: float(M.mean()) for S, M in sink_by_len.items()}
layer_prof_by_len = {S: M.mean(axis=1) for S, M in sink_by_len.items()}

rows = []
for S, M in sink_by_len.items():
    L, H = M.shape
    for l in range(L):
        for h in range(H):
            rows.append({'length': S, 'layer': l, 'head': h, 'sink_score': float(M[l, h])})
pd.DataFrame(rows).to_csv(EXP / 'sink_by_length.csv', index=False)
pd.DataFrame([{'length': S, 'global_sink': globals_by_len[S],
               'peak_layer': int(layer_prof_by_len[S].argmax())} for S in sorted(sink_by_len)]
             ).to_csv(EXP / 'summary_stats.csv', index=False)

U.plot_length_curves(globals_by_len, EXP / 'figures', 'length_curves.png')
U.plot_layer_progression_by_length(layer_prof_by_len, EXP / 'figures', 'layer_progression_by_length.png')

xs = sorted(globals_by_len)
print('Hypothesis check (sink increases with length):')
print('  global sink by length:', {S: round(globals_by_len[S], 4) for S in xs})
if len(xs) >= 2:
    print('  monotonic increasing over fitted lengths:', all(globals_by_len[xs[i]] <= globals_by_len[xs[i+1]] for i in range(len(xs)-1)))
if skipped:
    print('  NOTE: skipped', sorted(skipped), '(memory-gated; rerun on A100/high-RAM to include).')

## 5. Metric Validation  *(part of Experiment 2C)*

Experiment 2C found that the global sink score **decreases** as context length grows,
contradicting the hypothesis. Before concluding that sinks genuinely weaken, this
subsection tests whether the decrease is an artefact of the **metric** rather than a
change in the underlying mechanism.

`mean_from_k` averages attention to token 0 over *all* query positions after `k`. As the
context grows, that average absorbs many additional later queries — each of which spreads
its softmax mass over more keys. So the mean can fall even if the per-position sink is
completely unchanged.

**Why this test is decisive here.** `make_length_controlled_prompts` builds **nested
prefixes** (the 512-token prompt starts with exactly the tokens of the 128-token prompt),
and attention is **causal** — query position *q* can only attend to keys ≤ *q*. Therefore
attention at every position *q* < min(length) is *mathematically identical* across all
context lengths. Two consequences we can check directly:

1. A **fixed-window** metric (first 128 valid queries only) should be **flat** across lengths.
2. The **query-position profiles** should **coincide exactly** in their overlapping range.

If both hold, the decrease is entirely an averaging effect. This does not modify the 2C
metric or design — it only strengthens the interpretation.

Outputs are saved under `results/phase2/experiment2C/metric_validation/`.

In [ ]:
# --- Metric Validation: setup -------------------------------------------------
import matplotlib.pyplot as plt

VAL = EXP / 'metric_validation'
(VAL / 'figures').mkdir(parents=True, exist_ok=True)

# Validation needs the FULL per-query sink column [L,H,S], not the [L,H] summary.
# Reuse the columns retained in §3; re-extract (memory-gated) only if unavailable.
if 'col_by_len' not in globals() or not col_by_len:
    col_by_len = {}
    for S in sorted(sink_by_len):
        ok, req, avail = U.can_fit_length(S, model, safety=cfg.mem_safety)
        if not ok:
            logger.warning('validation: length %d no longer fits in memory; skipping', S)
            continue
        col_by_len[S] = U.extract_sink_column(prompts[S], model)
        logger.info('validation: re-extracted sink column for length %d', S)

lengths_val = sorted(col_by_len)
print('lengths available for validation:', lengths_val)

# Persist the raw columns so the validation is reproducible without re-running the model.
np.savez_compressed(VAL / 'sink_columns.npz',
                    **{str(S): col_by_len[S].astype(np.float16) for S in lengths_val})
print('saved sink_columns.npz')

### Validation 1 — fixed-window comparison

In [ ]:
# --- Validation 1: fixed-window vs original mean_from_k -----------------------
WINDOW = 128   # fixed number of valid query positions, regardless of context length

rows = []
for S in lengths_val:
    col = col_by_len[S]                      # [L, H, S]
    n_valid = col.shape[2] - cfg.k           # queries remaining after the skipped prefix
    w = min(WINDOW, n_valid)                 # at S=128 only n_valid < WINDOW exist
    original = col[:, :, cfg.k:].mean(axis=2)            # identical to the 2C metric
    fixed    = col[:, :, cfg.k:cfg.k + w].mean(axis=2)   # first `w` valid queries only
    rows.append({'length': S, 'n_valid_queries': int(n_valid), 'window_used': int(w),
                 'original_mean_from_k': float(original.mean()),
                 'fixed_window_mean_from_k': float(fixed.mean())})

val1 = pd.DataFrame(rows)
val1['ratio_fixed_over_original'] = val1['fixed_window_mean_from_k'] / val1['original_mean_from_k']
val1.to_csv(VAL / 'fixed_window_comparison.csv', index=False)

# Consistency: the 'original' column must reproduce the §4 result exactly.
for r in rows:
    assert abs(r['original_mean_from_k'] - globals_by_len[r['length']]) < 1e-5, 'metric mismatch'

print(val1.round(5).to_string(index=False))
print('\nNote: at length %d the window is %d (only %d valid queries exist), so the two'
      % (lengths_val[0], val1['window_used'].iloc[0], val1['n_valid_queries'].iloc[0]))
print('metrics coincide there by construction - it is the anchor point of the comparison.')

In [ ]:
# --- Prefix-invariance check (causal masking + nested prefixes) ---------------
# Attention at query q < min(length) cannot depend on tokens that appear later,
# so the per-position sink columns must agree across context lengths.
ref_S = lengths_val[0]
ref = col_by_len[ref_S]
n_ref = ref.shape[2]

inv_rows = []
for S in lengths_val[1:]:
    d = float(np.abs(col_by_len[S][:, :, :n_ref] - ref).max())
    inv_rows.append({'length': S, 'compared_positions': int(n_ref), 'max_abs_diff_vs_%d' % ref_S: d})
inv = pd.DataFrame(inv_rows)
inv.to_csv(VAL / 'prefix_invariance_check.csv', index=False)

max_dev = float(inv.iloc[:, -1].max()) if len(inv) else 0.0
print(inv.to_string(index=False) if len(inv) else '(only one length available)')
print('\nlargest deviation over overlapping positions: %.2e' % max_dev)
print('-> per-position sink behaviour is PRESERVED across lengths'
      if max_dev < 1e-2 else
      '-> positions differ more than expected; inspect before interpreting')

In [ ]:
# --- Validation 1: comparison plot -------------------------------------------
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(val1['length'], val1['original_mean_from_k'], 'o-', lw=2, color='#d62728',
        label='original mean_from_k (all queries $\\geq k$)')
ax.plot(val1['length'], val1['fixed_window_mean_from_k'], 's--', lw=2, color='#1f77b4',
        label='fixed window (first %d valid queries)' % WINDOW)
ax.set_xscale('log', base=2)
ax.set_xticks(lengths_val); ax.set_xticklabels(lengths_val)
ax.set_xlabel('context length (tokens, log2)')
ax.set_ylabel('global sink score')
ax.set_title('Metric validation: does the decrease survive a fixed query window?')
ax.legend(); ax.grid(True, which='both', alpha=0.25)
U.savefig(VAL / 'figures', 'fixed_window_comparison.png'); plt.show()

### Validation 2 — query-position profile

In [ ]:
# --- Validation 2: query-position profile for a representative sink head ------
# Prefer the strongest head from the Experiment 2A baseline; fall back to 2C itself.
b = U.load_baseline(Path(RESULTS_ROOT) / 'experiment2A')
if b is not None:
    L_star, H_star = np.unravel_index(int(b['matrix'].argmax()), b['matrix'].shape)
    head_source = 'Experiment 2A baseline (strongest layer x head)'
else:
    M_ref = sink_by_len[max(lengths_val)]
    L_star, H_star = np.unravel_index(int(M_ref.argmax()), M_ref.shape)
    head_source = '2C longest-context matrix (2A baseline not found)'
L_star, H_star = int(L_star), int(H_star)
print('representative sink head: layer %d, head %d  [%s]' % (L_star, H_star, head_source))

prof_rows = []
for S in lengths_val:
    p = col_by_len[S][L_star, H_star, :]
    for q, v in enumerate(p):
        prof_rows.append({'length': S, 'query_pos': q, 'attn_to_token0': float(v)})
pd.DataFrame(prof_rows).to_csv(VAL / 'query_position_profile.csv', index=False)

cmap = plt.get_cmap('viridis')
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for i, S in enumerate(lengths_val):
    c = cmap(i / max(1, len(lengths_val) - 1))
    p = col_by_len[S][L_star, H_star, :]
    axes[0].plot(np.arange(len(p)), p, lw=1.6, color=c, label='len %d' % S)
    axes[1].plot(np.arange(len(p))[:WINDOW], p[:WINDOW], lw=1.8, color=c, label='len %d' % S)
axes[0].set_xscale('log'); axes[0].set_xlabel('query position (log scale)')
axes[0].set_title('Full range')
axes[1].set_xlabel('query position'); axes[1].set_title('First %d positions (overlap region)' % WINDOW)
for ax in axes:
    ax.set_ylabel('attention(query $\\rightarrow$ token 0)')
    ax.axvline(cfg.k, color='k', ls=':', lw=1, alpha=0.6)
    ax.legend(fontsize=8)
fig.suptitle('Attention to token 0 vs query position - layer %d, head %d' % (L_star, H_star), y=1.02)
U.savefig(VAL / 'figures', 'query_position_profile.png'); plt.show()

### Interpretation

In [ ]:
# --- Automatic interpretation -------------------------------------------------
o = val1.set_index('length')['original_mean_from_k']
f = val1.set_index('length')['fixed_window_mean_from_k']
S_lo, S_hi = lengths_val[0], lengths_val[-1]

orig_change  = (o[S_hi] - o[S_lo]) / o[S_lo] if o[S_lo] else float('nan')
fixed_change = (f[S_hi] - f[S_lo]) / f[S_lo] if f[S_lo] else float('nan')
explained = (1 - abs(fixed_change) / abs(orig_change)) if orig_change else float('nan')

# Does the per-position sink itself decay with query position (within one length)?
p_hi = col_by_len[S_hi][L_star, H_star, :]
early = float(p_hi[cfg.k:cfg.k + WINDOW].mean())
late  = float(p_hi[-WINDOW:].mean())
decay_ratio = late / early if early else float('nan')

if abs(fixed_change) < 0.15 and orig_change < -0.15:
    verdict = ('METRIC ARTEFACT. The sink is largely PRESERVED. Holding the query window '
               'fixed removes the decrease, so the fall in the original score is driven by '
               'averaging over the many extra query positions that longer contexts add.')
elif fixed_change < -0.15:
    verdict = ('GENUINE WEAKENING. The decrease persists even within a fixed query window, '
               'so it is not merely an averaging effect.')
else:
    verdict = ('MIXED. The fixed-window metric changes only moderately; part of the decrease '
               'is an averaging effect and part may be genuine. Interpret with caution.')

lines = [
    'METRIC VALIDATION - INTERPRETATION',
    '=' * 66,
    'context lengths compared      : %s -> %s' % (S_lo, S_hi),
    'original mean_from_k change   : %+.1f%%  (%.4f -> %.4f)' % (100 * orig_change, o[S_lo], o[S_hi]),
    'fixed-window (%d q) change   : %+.1f%%  (%.4f -> %.4f)' % (WINDOW, 100 * fixed_change, f[S_lo], f[S_hi]),
    'share of drop explained by extra query positions : %.1f%%' % (100 * explained),
    'max per-position deviation over overlap          : %.2e' % max_dev,
    '',
    'within the longest context (layer %d, head %d):' % (L_star, H_star),
    '  mean attention to token 0, first %d queries : %.4f' % (WINDOW, early),
    '  mean attention to token 0, last  %d queries : %.4f' % (WINDOW, late),
    '  late/early ratio                            : %.3f' % decay_ratio,
    '',
    'VERDICT: ' + verdict,
    '',
    'Caveats: the shortest length is the anchor point (its window equals its own query',
    'count, so the two metrics coincide there by construction); and per-query softmax mass',
    'is spread over more keys as position grows, which mechanically dilutes any single',
    'column. This subsection tests the METRIC, not the mechanism - causal explanations',
    'remain out of scope for Phase 2.',
]
report = chr(10).join(lines)
print(report)
(VAL / 'interpretation.txt').write_text(report, encoding='utf-8')

import json
with open(VAL / 'interpretation.json', 'w') as fh:
    json.dump({'lengths': lengths_val, 'window': WINDOW,
               'original_change_frac': float(orig_change),
               'fixed_window_change_frac': float(fixed_change),
               'share_explained_by_extra_queries': float(explained),
               'max_abs_deviation_overlap': float(max_dev),
               'representative_layer': L_star, 'representative_head': H_star,
               'head_source': head_source,
               'early_mean': early, 'late_mean': late, 'late_over_early': float(decay_ratio),
               'verdict': verdict}, fh, indent=2)
print('\nsaved interpretation.txt / interpretation.json ->', VAL)

## Download results

In [ ]:
# --- Download this experiment's outputs -------------------------------------
import shutil
exp = Path(RESULTS_ROOT) / 'experiment2C'
zp = shutil.make_archive(str(Path('/content' if IN_COLAB else '.') / ('experiment2C_outputs')), 'zip', exp)
print('Bundled:', zp)
if IN_COLAB:
    from google.colab import files
    files.download(zp)
